In [2]:
%load_ext autoreload
%autoreload 2
import numpy as np
import emcee
import corner
from matplotlib import pyplot as plt
from astropy import units as u
from astropy import constants as const
import time
from multiprocessing import Pool

from nfw_profile_heating import NFWProfile
from cluster_error_prop import Cluster_werr
from statistics_functions import log_lik
from load_data_and_var import load_data_and_var, generate_nfws_from_combined_dataset_werr
from plotting import savefig, paper_plot

In [3]:
dataset, variances = load_data_and_var()

In [77]:
nfws_werr = generate_nfws_from_combined_dataset_werr(dataset, variances)

In [78]:
nfws= [nfw for nfw in nfws_werr if nfw.energies.is_virialized_gaspari]
len(nfws)

111

In [79]:
test_nfw = nfws[5]

In [80]:
test_nfw.M500.value

354288799999999.94

In [92]:
test_nfw.M500.plus

33280699999999.996

In [81]:
s0 = 1e-27
mx=1e-2

In [90]:
def estimate_variance_monte_carlo(nfw, mx, s0, num_samples=1000):
    # Means and standard deviations for the parameters
    mLinj_mean = 0.38
    mLinj_std = 0.03
    bLinj_mean = 10.0
    bLinj_std = 0.11
    mT500_mean = 4.71
    mT500_std = 0.26
    bT500_mean = -2.34
    bT500_std = 0.09
    
    M500_mean = nfw.M500.value
    M500_std = max(nfw.M500.plus, nfw.M500.minus)

    L500_samples = []
    for _ in range(num_samples):
        #mLinj_sample = np.random.normal(mLinj_mean, mLinj_std)
        #bLinj_sample = np.random.normal(bLinj_mean, bLinj_std)
        #mT500_sample = np.random.normal(mT500_mean, mT500_std)
        #bT500_sample = np.random.normal(bT500_mean, bT500_std)
        M500_sample = np.random.normal(M500_mean, M500_std)
        
        scaling_params={
                'mLinj': np.random.normal(mLinj_mean, mLinj_std),
                'bLinj': np.random.normal(bLinj_mean, bLinj_std),
                'mT500': np.random.normal(mT500_mean, mT500_std),
                'bT500': np.random.normal(bT500_mean, bT500_std)
        }
            
        print(scaling_params)
        print(M500_sample)

        L500_sample = nfw.precalc.L500_numerical_sampled(M500_sample, mx, s0, scaling_params)
        L500_samples.append(L500_sample)

    L500_samples = np.array(L500_samples)
    uncertainty = np.std(L500_samples)
    return uncertainty

In [91]:
unc_27_2 = estimate_variance_monte_carlo(test_nfw, mx, s0)

{'mLinj': 0.36210621163178863, 'bLinj': 10.064170755630816, 'mT500': 4.484097035812248, 'bT500': -2.2235320004532944}
350811317993513.7
{'mLinj': 0.41566161264203283, 'bLinj': 9.896625460783904, 'mT500': 4.716793001841514, 'bT500': -2.3901302052078837}
344137441995489.9
{'mLinj': 0.3572803444872206, 'bLinj': 9.901597400997918, 'mT500': 4.776901117890419, 'bT500': -2.184607260811149}
346198480683538.8
{'mLinj': 0.39667646876403745, 'bLinj': 10.417057531011618, 'mT500': 5.121752259180102, 'bT500': -2.250405928720126}
378368517765045.5
{'mLinj': 0.32580685848107993, 'bLinj': 10.001937804475752, 'mT500': 4.794572302814663, 'bT500': -2.4506066501394552}
338874876968744.2
{'mLinj': 0.3814975278725524, 'bLinj': 9.95613926262929, 'mT500': 4.767564082376641, 'bT500': -2.3381107786035367}
366963407715710.0
{'mLinj': 0.33507936389234083, 'bLinj': 9.854133117456701, 'mT500': 4.77312151217621, 'bT500': -2.3862747924162235}
378008313818652.44
{'mLinj': 0.40434943508566407, 'bLinj': 9.854585770241664

ValueError: f(a) and f(b) must have different signs

In [84]:
var_27_2

array([ 119.02155652,    6.30479751,    6.42293223,   13.74422526,
         25.77641678,   27.99431439,    6.90615743,    3.95417616,
        187.95025187,  448.91578471,   47.3384943 ,   27.95119349,
       1173.91398219,    2.17611081,   10.61460423,   28.71881204,
         12.0143731 ,   46.75592283,  523.46804304,   67.60150439,
        140.71698602,  189.02804965,   19.72802922,   81.77739334,
          8.53777451,  215.98364027,   19.4665692 ,   22.51447922,
        179.42549591,   53.70692259,   11.5841136 ,    4.9721751 ,
         22.90333217,    5.30217982,  123.77002589,  100.52501092,
         49.80539761,   16.76492412,   25.6171913 ,  661.72907622,
        235.40593808,   35.11310983,  145.95942388,   11.44588065,
         15.18813855,   55.10877729,   36.02892335,  394.6396744 ,
        172.48420293,   10.26591153,   23.40825913,   12.21205993,
         38.18855841,    3.1025356 ,   44.08813978,    7.69948113,
         56.96479711,   28.59963137,   15.21597628,   17.37302

In [87]:
np.std(var_27_2)

155.6376493084629